# Benchmark - run a grid on Colab
Plays the games of a grid (`grids/*.yaml`) that have not been played yet, one model at a time, and saves one record per game on Google Drive.
The games already played are skipped: the new ones on Drive and the old experiments imported in `results/` (see `import_legacy.py`).
If the run stops (or the runtime is deleted), run the notebook again: it resumes from the games not yet saved.

Recommended order: a pilot of a few games per new model (`limit = 8`) to see the speed on the L4, then the whole `tier1.yaml`, then `tier1_families.yaml`.

# Setup
Clones the repository (only the folders needed), installs TextWorld and vLLM and moves into `Code/Benchmark`. Locally it does nothing.
The first time on a new Colab machine the installation takes a few minutes: restart the kernel after it (and then run all the cells), because it can update torch.

In [ ]:
import os
import sys
import shutil
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_url = "https://github.com/UPROOT01/LLMs-Play-Textual-Games.git"
    repo_dir = "/content/LLMs-Play-Textual-Games"
    if not os.path.isfile(f"{repo_dir}/Code/Benchmark/run.py"):
        shutil.rmtree(repo_dir, ignore_errors=True) # leftover of a failed clone, or a clone without the benchmark
        # sparse clone: only the folders needed (the whole Games folder is >1 GB)
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", repo_url, repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    subprocess.run(["git", "-C", repo_dir, "sparse-checkout", "add", "Code/Benchmark",
                    "Code/Games/cooking", "Code/Games/coin_collector", "Code/Games/treasure_hunter"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "textworld", "vllm==0.31.0", "pyyaml"], check=True)
    os.chdir(f"{repo_dir}/Code/Benchmark")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU available: connect to a Colab runtime with a GPU (e.g. L4)"
! nvidia-smi

# What to run
- `grid`: the grid file. `models`: only these models of the grid (`None`: all of them, one after the other).
- `limit`: at most this many games per model (`None`: all). Use it for a pilot.
- `n_parallel`: games played at the same time. vLLM answers them together: more games, more tokens per minute, until the GPU memory is full.

To change the models or their vLLM settings (e.g. a smaller `max_model_len` if a model does not fit), see `profiles.py`.

In [ ]:
from records import LEGACY_RESULTS
from run import load_grid, run_grid, summary

grid = load_grid("grids/tier1.yaml")
models = ["Qwen/Qwen3-1.7B"] # None: all the models of the grid
limit = None # e.g. 8 for a pilot
n_parallel = 8

if IN_COLAB:
    assert os.path.isdir("/content/drive/MyDrive"), "Google Drive is not mounted: run the cell above"
    results = "/content/drive/MyDrive/LLMs-Play-Textual-Games/Benchmark/results"
else:
    results = "results_new"
read = [LEGACY_RESULTS] # the old experiments: their games are not played again
print("Results folder:", results)
summary(grid, results, read, models)

# Run
Every game is saved as soon as it ends. Stopping the cell is safe: the games being played are lost, the others are kept.
If loading the second model of a run fails because the GPU memory is full, restart the runtime and run everything again: the first model's games are skipped.

In [ ]:
run_grid(grid, results, read, models=models, limit=limit, n_parallel=n_parallel)

# Quick look
Average normalized score, games and thousands of generated tokens per game, per model and mode (the graphs are in `analysis.ipynb`).

In [ ]:
from records import load_table

table = load_table(results, LEGACY_RESULTS)
games = table[table.game.isin(grid["games"]) & table.prompt.isin(grid["prompts"]) & table.seed.isin(grid["seeds"])
              & table.model.isin(grid["models"]) & table["mode"].isin(grid["modes"])]
games = games[[level in grid["games"][game] for game, level in zip(games.game, games.level)]]
games.groupby(["model", "mode"]).agg(score=("norm_score", "mean"), games=("seed", "size"),
                                     ktokens=("tokens_generated", lambda t: t.mean() / 1000)).round(2)